# Outbound Auth

Outbound Auth를 사용하면 에이전트와 AgentCore Gateway가 Inbound Auth에서 인증 및 권한 부여된 사용자를 대신해 AWS 리소스와 서드 파티 서비스에 안전하게 액세스할 수 있습니다. AWS 리소스 또는 서드 파티 서비스와 권한 부여를 통합하려면 Inbound Auth와 Outbound Auth를 모두 구성해야 합니다.

AgentCore Identity가 지원하는 최소 필요 액세스와 안전한 권한 위임을 통해 에이전트는 AWS 리소스 및 GitHub, Google, Salesforce, Slack 같은 서드 파티 도구에 원활하고 안전하게 액세스할 수 있습니다. 사전에 사용자의 동의를 받았다면 에이전트는 사용자를 대신하거나 독립적으로 이러한 서비스에서 작업을 수행할 수 있습니다. 또한 안전한 token vault를 사용해 반복적인 동의 요청을 줄이고 간소화된 AI 에이전트 경험을 만들 수 있습니다.

## Outbound Auth 구성

먼저 서드 파티 provider에 클라이언트 애플리케이션을 등록한 다음 Outbound Auth를 생성합니다. AWS 리소스, 서드 파티 서비스 또는 AgentCore Gateway 대상에 대한 액세스를 검증할 방법을 지정합니다. OAuth 2LO/3LO 또는 API key를 사용할 수 있습니다. OAuth를 사용할 때는 AgentCore Identity가 제공하는 provider를 선택하고 해당 구성 정보를 입력하거나, custom provider의 세부 정보를 직접 제공할 수 있습니다. 

사용자가 AWS 리소스, 서드 파티 서비스 또는 AgentCore Gateway 대상에 액세스하려고 하면 Outbound Auth가 Inbound Auth에서 제공한 access token의 유효성을 확인하고, 유효한 경우 리소스 액세스를 허용합니다.

<div style="text-align:center">
    <img src="images/outbound_auth.png" width="90%"/>
</div>


다음은 `@require_access_token` decorator에서 사용할 수 있는 파라미터입니다.


| 파라미터 이름       | 설명                                                                     |
|:--------------------|:-------------------------------------------------------------------------|
| provider_name       | credential provider 이름                                                 |
| into                | 토큰을 주입할 파라미터 이름                                              |
| scopes              | 요청할 OAuth2 scope                                                      |
| on_auth_url	      | authorization URL 처리용 callback                                        |
| auth_flow           | 인증 흐름 유형("M2M" 또는 "USER_FEDERATION")                            |
| callback_url        | OAuth2 callback URL                                                      |
| force_authentication| 재인증 강제                                                               |
| token_poller        | custom token poller 구현                                                  |

		


# Amazon Bedrock AgentCore Runtime에 Strands Agents 호스팅

## 개요


이 튜토리얼에서는 사용자의 Google Calendar 이벤트를 조회하는 일정 관리 에이전트를 Strands Agents로 개발합니다. Google 자격 증명을 관리할 credential provider를 구성하고, 에이전트 코드에서 해당 provider를 호출해 `access_token`을 사용하여 사용자의 Calendar 이벤트나 일정을 가져오도록 수정합니다.

### 튜토리얼 아키텍처

<div style="text-align:center">
    <img src="images/outbound_auth_3lo.png" width="90%"/>
</div>


### 튜토리얼 세부 정보

| 항목                | 세부 정보                                                                |
|:--------------------|:-------------------------------------------------------------------------|
| 튜토리얼 유형       | 대화형                                                                   |
| 에이전트 유형       | 단일                                                                     |
| Agentic Framework   | Strands Agents                                                           |
| LLM model           | Anthropic Claude Haiku 4.5                                              |
| 튜토리얼 구성 요소  | AgentCore Runtime에 에이전트 호스팅, Strands Agent 및 Claude 모델 사용   |
| 튜토리얼 분야       | 산업 공통                                                                |
| 예제 난이도         | 중간                                                                     |
| 사용 SDK            | Amazon BedrockAgentCore Python SDK 및 boto3                              |
| Credential Provider | 유형: OAuth2 - Google Provider                                           |


### 튜토리얼 주요 기능

* Amazon Bedrock AgentCore Runtime에 에이전트 호스팅
* Claude 모델 사용
* Strands Agents 사용
* OAuth2 Google credential provider를 이용한 AgentCore Outbound Auth 사용


## 사전 요구 사항

이 튜토리얼을 실행하려면 다음 항목이 필요합니다.
* Python 3.10+
* AWS 자격 증명
* Amazon Bedrock AgentCore SDK
* Strands Agents
* 실행 중인 Docker

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

## Cognito를 IdP로 사용하는 Inbound Auth 구성
App client와 테스트 사용자 한 명이 포함된 Cognito User Pool을 프로비저닝하겠습니다. 배포된 MCP server에 액세스할 수 있도록 Amazon Cognito에서 JWT 토큰을 발급합니다. 이를 위해 `utils` 스크립트의 `setup_cognito_user_pool` 도우미 함수를 사용합니다.

참고: Cognito `access_token`의 유효 시간은 2시간입니다. 만료된 경우 `reauthenticate_user` 메서드로 새 `access_token`을 발급할 수 있습니다.

In [ ]:
import sys
import os

# 현재 Notebook의 디렉터리 확인
current_dir = os.path.dirname(os.path.abspath("__file__" if "__file__" in globals() else "."))

utils_dir = os.path.join(current_dir, "..")
utils_dir = os.path.abspath(utils_dir)

# sys.path에 추가
sys.path.insert(0, utils_dir)
print("sys.path[0]:", sys.path[0])

In [ ]:
import subprocess
from boto3.session import Session
from utils import setup_cognito_user_pool, reauthenticate_user

boto_session = Session()
region = boto_session.region_name

print(f"Region: {region}")

identity_client = boto_session.client("bedrock-agentcore-control")

In [ ]:
region

In [ ]:
print("Setting up Amazon Cognito user pool...")
cognito_config = setup_cognito_user_pool("Cognito_3LO_Google")
print("Cognito setup completed ✓")

## Google OAuth2 구성(On-behalf-of-user 흐름)
다음 단계에 따라 앱을 등록하고 프로젝트를 생성한 뒤 Google Calendar 액세스를 위한 OAuth 자격 증명을 구성하세요.

이 섹션에서는 읽기 전용 권한으로 Google Calendar API에 액세스하도록 Google을 구성합니다.

1. Google Developer Console에서 프로젝트 생성
    1.    [Google Developer Console](https://console.developers.google.com/)로 이동합니다.
    2.    상단 탐색 모음에서 “Create Project”를 클릭합니다.
    3.    Project Name을 입력합니다.
    4.    Organization을 선택하거나 해당하지 않으면 “No organization”을 유지합니다.
    5.    Create를 클릭합니다. 새 프로젝트가 프로젝트 목록에 표시됩니다.
2. Google Calendar API 활성화
    1.    체크박스로 프로젝트를 선택한 상태에서 왼쪽 메뉴를 열고 APIs & Services > Library로 이동합니다.
    2.    검색창에 Google Calendar API를 입력합니다.
    3.    결과에서 Google Calendar API를 선택한 다음 Enable을 클릭합니다.
3. OAuth Consent Screen 구성
    1.    왼쪽 메뉴에서 APIs & Services > OAuth consent screen으로 이동합니다.
    2.    “Get started”를 클릭합니다.
    3.    다음 필수 항목을 입력합니다.
    4.    App Name
    5.    User Support Email
    6. Next를 클릭하고 적절한 Audience 유형(Internal 또는 External)을 선택한 뒤 Next를 클릭합니다. External을 선택했다면 테스트할 사용자의 이메일 주소를 입력해야 합니다.
    7.    Developer Contact Information에 이메일 주소를 입력합니다.
    8.  이용 약관에 동의한 후 “Finish”를 클릭하고 “Create”를 클릭합니다.
4. Google 이메일 주소를 테스트 사용자로 추가
    1.    왼쪽 메뉴에서 APIs & Services > OAuth consent screen으로 이동합니다.
    2.    왼쪽 메뉴에서 "Audience"를 선택합니다.
    3.    Test users에서 "+ Add Users"를 클릭하고 Google 계정의 Gmail 주소를 추가합니다.
5. OAuth 2.0 자격 증명 생성
    1.    왼쪽 메뉴에서 APIs & Services > Credentials로 이동합니다.
    2.    Create Credentials > OAuth client ID를 클릭합니다.
    3.    애플리케이션 유형으로 Web application을 선택합니다.
    4.    자격 증명 이름을 입력합니다.
    5.    Create를 클릭합니다.
6. Client ID 및 Client Secret 확인
    1.    생성 후 대화 상자에 Client ID와 Client Secret이 표시됩니다. 나중에 사용할 수 있도록 복사합니다.
    2.    자격 증명을 JSON 파일로 다운로드하거나 앱 구성에 사용할 수 있도록 복사합니다.
7. Data access 업데이트
    1. 왼쪽 메뉴에서 APIs & Services > Credentials로 이동합니다.
    2. 앞 단계에서 생성한 “web app”을 선택합니다.
    3. 왼쪽 메뉴에서 “Data access”를 선택합니다.
    4. “Add or remove scopes”를 클릭합니다.
    5. 사용 사례에 맞는 scope를 추가합니다. 예를 들어 Google Calendar의 경우 “Manually add scopes”에 “https://www.googleapis.com/auth/calendar.readonly”를 추가하고 update를 클릭한 뒤 Save/Update를 클릭합니다.
    6. “Data access” 페이지에서 “Save”를 다시 클릭하여 구성을 저장합니다.
8. 에이전트에서 자격 증명 사용
    1.    다음 섹션에서는 OAuth 3-legged 흐름에 Client ID, Client Secret, Redirect URI를 사용하도록 resource credential provider를 구성합니다.

## OAuth2 Authorization URL Session Binding 과정

OAuth2 authorization URL session binding은 OAuth2 권한 부여 session을 AgentCore Identity의 인증된 사용자와 올바르게 연결하는 핵심 보안 메커니즘입니다. 이 과정은 session 탈취를 방지하고 OAuth 토큰이 의도한 사용자에게만 발급되도록 보장합니다.

참고: https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/oauth2-authorization-url-session-binding.html

### Session binding 작동 방식
<div style="text-align:center">
    <img src="images/identity-session-binding.png" width="90%"/>
</div>

1. 에이전트 호출: 에이전트 사용자가 자신이 소유한 애플리케이션 또는 리소스에 액세스하려고 하면 에이전트 코드가 `GetResourceOauth2Token` API를 호출하여 authorization URL을 가져옵니다.

2. authorization URL 생성: AgentCore Identity가 사용자가 이동하여 액세스에 동의할 authorization URL과 session URI를 생성합니다.

3. 권한 부여 및 access token 획득: 사용자가 authorization URL로 이동해 에이전트의 리소스 액세스에 동의합니다. 이후 AgentCore Identity는 권한 부여 요청을 시작한 사용자 정보와 함께 사용자의 브라우저를 HTTPS 애플리케이션 endpoint로 리디렉션합니다. 이 endpoint는 요청을 시작한 에이전트 사용자와 현재 애플리케이션에 로그인한 사용자가 동일한지 확인합니다. 일치하면 애플리케이션 endpoint가 `CompleteResourceTokenAuth`를 호출하여 AgentCore Identity가 access token을 가져와 저장하도록 합니다.

4. access token을 얻기 위해 에이전트 재호출: 애플리케이션이 유효한 응답을 반환하면 에이전트 애플리케이션이 해당 사용자에 대해 처음 요청한 OAuth2.0 access token을 가져올 수 있습니다. 사용자가 일치하지 않으면 애플리케이션은 아무 작업도 하지 않거나 시도를 기록합니다.

애플리케이션 endpoint에서 사용자 신원을 검증하므로, AgentCore Identity는 권한 부여 요청을 시작한 사용자와 액세스에 동의한 사용자가 항상 동일한지 에이전트 애플리케이션이 확인할 수 있게 합니다.

### 이 샘플의 OAuth2 Session Binding 흐름 개요

OAuth2 session binding 과정은 애플리케이션, AgentCore Identity, 외부 OAuth provider(예: Google, GitHub), 로컬 callback server 사이를 조정하는 다음 핵심 단계로 구성됩니다.

#### 1단계: 애플리케이션 Callback URL 생성
- 애플리케이션에 공개적으로 액세스할 수 있는 HTTPS callback endpoint를 생성합니다.
- 이 endpoint가 OAuth 리디렉션을 처리하고 사용자 session을 검증합니다.
- 예: `https://myagentapp.com/callback`

#### 2단계: Callback URL로 Workload Identity 업데이트
- callback URL을 workload identity의 `AllowedResourceOauth2ReturnUrl`로 등록합니다.
- `UpdateWorkloadIdentity` API를 사용해 등록합니다.
- **이 튜토리얼에서는** 아래 코드가 로컬 callback server URL로 workload identity를 업데이트하여 이 단계를 자동으로 처리합니다.

#### 3단계: OAuth2 Credential Provider 생성
- 외부 OAuth provider 정보(client ID, secret)로 credential provider를 구성합니다.
- AgentCore Identity가 provider의 고유 callback URL을 반환합니다.
- 이 callback URL을 외부 OAuth provider(예: Google Console)에 등록합니다.

#### 4단계: Session 검증 및 토큰 완료 구현
- callback endpoint에서 현재 사용자의 session을 검증해야 합니다.
- 사용자 식별자와 session URI로 `CompleteResourceTokenAuth` API를 호출합니다.
- **이 튜토리얼에서는** `oauth2_callback_server.py`가 이 작업을 자동으로 처리합니다.

#### 5단계: OAuth 흐름 실행
- 사용자가 에이전트와 상호 작용하여 OAuth 흐름을 시작합니다.
- 권한 부여를 위해 사용자가 외부 provider로 리디렉션됩니다.
- provider가 session 정보와 함께 사용자를 callback으로 다시 리디렉션합니다.
- session binding이 완료되고 OAuth 토큰을 사용할 수 있게 됩니다.

### oauth2_callback_server.py를 사용한 로컬 개발

이 튜토리얼에서는 로컬 개발 및 테스트를 위해 다음 기능을 제공하는 `oauth2_callback_server.py`를 사용합니다.

1. **로컬 FastAPI Server 실행**(`localhost:9090`)
   - 상태 확인용 `/ping` endpoint 제공
   - 사용자 토큰 저장용 `/userIdentifier/token` endpoint 제공
   - OAuth 리디렉션 처리용 `/oauth2/callback` endpoint 제공

2. **사용자 토큰 저장소 관리**
   - Cognito 인증에서 받은 사용자의 JWT 토큰 저장
   - OAuth session을 올바른 사용자 신원과 연결

3. **OAuth Callback 처리**
   - `session_id` 파라미터가 포함된 OAuth 리디렉션 수신
   - `CompleteResourceTokenAuth`를 호출해 session 바인딩
   - 흐름을 완료하기 전에 사용자 신원 검증

4. **Session 보안 제공**
   - OAuth session이 인증된 사용자와 바인딩되도록 보장
   - OAuth 토큰에 대한 무단 액세스 방지

### Workload Identity 업데이트와의 통합

다음 코드 조각은 OAuth2 session binding 과정의 핵심 단계를 수행합니다.

```python
if launch_result.agent_id:
    workload_name = launch_result.agent_id
    workload_identity = identity_client.get_workload_identity(name=workload_name)
    allowed_resource_oauth_2_return_urls = workload_identity.get("allowedResourceOauth2ReturnUrls") or []
    oauth2_callback_url = get_oauth2_callback_url()
    print(f"Updating workload {workload_name} with callback url {oauth2_callback_url}")

    updated_workload_identity = identity_client.update_workload_identity(
        name=workload_name,
        allowed_resource_oauth_2_return_urls=[*allowed_resource_oauth_2_return_urls, oauth2_callback_url],
    )
```

이 코드는 다음 작업을 수행합니다.
1. **에이전트의 Workload Identity 조회**: Runtime 배포의 agent ID 사용
2. **현재 허용된 URL 확인**: 기존 `allowedResourceOauth2ReturnUrls` 조회
3. **로컬 Callback URL 추가**: `http://localhost:9090/oauth2/callback`을 허용된 반환 URL로 포함
4. **Workload Identity 업데이트**: AgentCore Identity에 callback URL 등록

AgentCore Identity는 사전 등록된 URL로만 OAuth callback을 리디렉션하므로 이 등록은 추가 보안 계층을 제공하는 데 필수적입니다.

### 보안 고려 사항

OAuth2 session binding 과정에는 다음 보안 조치가 포함됩니다.
- **URL 검증**: 사전 등록된 callback URL만 허용
- **Session 검증**: 토큰 완료 전에 사용자 session 검증 필요
- **사용자 신원 바인딩**: OAuth session을 인증된 사용자에게 명시적으로 바인딩
- **토큰 격리**: 각 사용자의 OAuth 토큰을 안전하게 격리

이 포괄적인 접근 방식은 다중 사용자 환경에서 OAuth2 흐름의 보안을 유지하고 올바른 사용자에게 정확히 귀속되도록 합니다.

---

### Google OAuth2 credential provider 구성

현재 폴더에 `.env` 파일을 생성하고 다음 내용을 복사하세요.
```sh
GOOGLE_CLIENT_ID="" # 위 6.1단계에서 기록한 "client id"
GOOGLE_CLIENT_SECRET="" # 위 6.2단계에서 기록한 "client secret"
```

client ID와 client secret을 업데이트한 다음 아래 코드를 실행해 Google용 credential provider를 생성하세요. <br>
AgentCore Identity의 resource credential provider는 에이전트, IdP, resource server 사이의 복잡한 관계를 관리하는 지능형 중개자 역할을 합니다. 각 provider는 특정 서비스 또는 ID 시스템에 필요한 endpoint 구성을 캡슐화합니다. 이 서비스는 개발 노력을 줄일 수 있도록 Google, GitHub, Slack, Salesforce 같은 널리 사용되는 서비스에 대해 authorization server endpoint와 provider별 파라미터가 미리 구성된 built-in provider를 제공합니다. 또한 OAuth2 호환 resource server에 맞게 조정할 수 있는 OAuth2 credential provider를 통해 custom 구성을 지원합니다.


In [ ]:
%%writefile .env
GOOGLE_CLIENT_ID="" # 위 6.1단계에서 기록한 "client id"
GOOGLE_CLIENT_SECRET="" # 위 6.2단계에서 기록한 "client secret"

In [ ]:
import dotenv

dotenv.load_dotenv(override=True)

# Google OAuth2 provider 구성 - 사용자 대신 작업
google_provider = identity_client.create_oauth2_credential_provider(
    **{
        "name": "google-cal-provider",
        "credentialProviderVendor": "GoogleOauth2",
        "oauth2ProviderConfigInput": {
            "googleOauth2ProviderConfig": {
                "clientId": os.environ["GOOGLE_CLIENT_ID"],
                "clientSecret": os.environ["GOOGLE_CLIENT_SECRET"],
            }
        },
    }
)
print(google_provider)
print("\n")
print(f"callbackUrl: {google_provider['callbackUrl']}")

## Google/OAuth 2.0 client의 callback URL 업데이트
[Google Developer Console](https://console.developers.google.com/)로 돌아갑니다.

1. Google Developer Console에서 프로젝트 선택
    1.    앞에서 생성한 프로젝트를 선택합니다.
2. callback URI 업데이트
    1.    왼쪽 메뉴에서 APIs & Services > Credentials로 이동합니다.
    2.    "OAuth 2.0 Client IDs" 아래의 client를 클릭합니다.
    3.    "Authorised redirect URIs"에 이전 단계의 callback URL을 입력합니다. 앞 단계에서 출력된 URL을 복사하면 됩니다.
    4.    Save를 클릭합니다.

## AgentCore Runtime 배포를 위한 에이전트 준비

### Amazon Bedrock에 호스팅된 모델을 사용하는 Strands Agent
다음 Strands Agent 코드에는 아래 기능이 포함되어 있습니다.
1. 오늘의 Google Calendar 이벤트를 가져오는 `get_calendar_events_today` 도구 생성
2. 앞 단계에서 생성한 credential provider를 사용해 Google에서 `access_token` 가져오기. 이 과정에는 3LO 흐름의 일부로 사용자에게 동의를 요청하고 승인을 받는 user consent 흐름이 포함됩니다.
3. 사용자 일정과 관련된 요청에 대해 Strands Agent가 도구 호출

In [ ]:
# 현재 환경(Notebook/SageMaker)에 맞는 OAuth2 callback URL 확인
# 에이전트 컨테이너가 아니라 이 Notebook에서 평가됨
from oauth2_callback_server import get_oauth2_callback_url

oauth2_callback_url_for_agent = get_oauth2_callback_url()

print(f"Callback URL for agent (determined from notebook environment): {oauth2_callback_url_for_agent}")

## AgentCore Runtime에 에이전트 배포
`CreateAgentRuntime` 작업은 컨테이너 이미지, 환경 변수, 암호화 설정 등 폭넓은 구성 옵션을 지원합니다. 프로토콜 설정(HTTP, MCP)과 권한 부여 메커니즘도 구성하여 클라이언트가 에이전트와 통신하는 방식을 제어할 수 있습니다.

참고: 운영 환경에서는 코드를 컨테이너로 패키징하고 CI/CD 파이프라인과 IaC를 사용해 ECR에 푸시하는 것이 좋습니다.

이 튜토리얼에서는 Amazon Bedrock AgentCore Python SDK로 아티팩트를 간편하게 패키징하고 AgentCore Runtime에 배포합니다.


### AgentCore Runtime 배포 구성

이제 starter toolkit을 사용해 엔트리포인트, 앞에서 생성한 실행 역할, requirements 파일로 AgentCore Runtime 배포를 구성합니다. 시작 시 Amazon ECR 리포지토리를 자동 생성하도록 starter toolkit도 구성합니다.

구성 단계에서는 애플리케이션 코드를 기반으로 Dockerfile이 생성됩니다.

참고: `authorizer_configuration`은 Cognito를 사용하는 Inbound Auth로 구성됩니다.

<div style="text-align:left">
    <img src="images/configure.png" width="40%"/>
</div>

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

print(f"Region: {region}")

discovery_url = cognito_config.get("discovery_url", "")
client_id = cognito_config.get("client_id", "")
agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="strands_claude_google_3lo.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    memory_mode="NO_MEMORY",
    agent_name="strands_agent_google_3lo",
    authorizer_configuration={
        "customJWTAuthorizer": {
            "discoveryUrl": discovery_url,
            "allowedClients": [client_id],
        }
    },
)
print(response)

## AgentCore 구성 검토

In [ ]:
!cat .bedrock_agentcore.yaml

### AgentCore Runtime에 에이전트 시작

Dockerfile이 준비되었으므로 에이전트를 AgentCore Runtime에 시작하겠습니다. 이 과정에서 Amazon ECR 리포지토리와 AgentCore Runtime이 생성됩니다.

<div style="text-align:left">
    <img src="images/launch.png" width="75%"/>
</div>

In [ ]:
from oauth2_callback_server import get_oauth2_callback_url

# AgentCore Runtime에 에이전트를 배포하고 배포 세부 정보 확인
launch_result = agentcore_runtime.launch(
    env_vars={"CALLBACK_URL": oauth2_callback_url_for_agent},
    auto_update_on_conflict=True,
)
print(launch_result)

if launch_result.agent_id:
    # ID 관리를 위해 배포된 에이전트 ID에서 workload 이름 추출
    workload_name = launch_result.agent_id
    # AgentCore Identity에서 현재 workload identity 구성 조회
    workload_identity = identity_client.get_workload_identity(name=workload_name)
    # 이 workload에 이미 등록된 OAuth2 callback URL 추출
    allowed_resource_oauth_2_return_urls = workload_identity.get("allowedResourceOauth2ReturnUrls") or []
    # session binding용 로컬 OAuth2 callback server URL 확인(localhost:9090/oauth2/callback)
    oauth2_callback_url = get_oauth2_callback_url()
    print(f"Updating workload {workload_name} with callback url {oauth2_callback_url}")

    # OAuth2 session binding을 활성화하도록 workload identity에 로컬 callback URL 등록
    updated_workload_identity = identity_client.update_workload_identity(
        name=workload_name,
        allowedResourceOauth2ReturnUrls=[
            *allowed_resource_oauth_2_return_urls,
            oauth2_callback_url,
        ],
    )
    print(updated_workload_identity)

#### 자동 생성된 역할에 필요한 추가 정책 연결

해당 모델에 액세스한 적이 없는 새 계정에서 실행하는 경우, 에이전트가 모델에 액세스할 수 있도록 자동 생성된 역할에 필요한 정책을 추가해야 합니다.

In [ ]:
import json
import boto3

agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)

runtime_response = agentcore_control_client.get_agent_runtime(agentRuntimeId=launch_result.agent_id)
runtime_role = runtime_response["roleArn"]
account = boto_session.client("sts").get_caller_identity().get("Account")

policies_to_add = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "BedrockModelAccess",
            "Effect": "Allow",
            "Action": [
                "aws-marketplace:ViewSubscriptions",
                "aws-marketplace:Subscribe",
            ],
            "Resource": "*",
        },
        {
            "Sid": "Oauth2TokenAccess",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:GetResourceOauth2Token",
            ],
            "Resource": "*",
        },
        {
            "Sid": "SecretsManagerAccess",
            "Effect": "Allow",
            "Action": [
                "secretsmanager:GetSecretValue",
            ],
            "Resource": [
                f"arn:aws:secretsmanager:{region}:{account}:secret:secret:bedrock-agentcore-identity!default/oauth2/google-cal-provider*"
            ],
        },
    ],
}
iam_client = boto3.client("iam", region_name=region)

response = iam_client.put_role_policy(
    PolicyDocument=json.dumps(policies_to_add),
    PolicyName="outbound_policies",
    RoleName=runtime_role.split("/")[1],
)

### AgentCore Runtime 상태 확인
AgentCore Runtime을 배포했으므로 배포 상태를 확인하겠습니다.

In [ ]:
import time

status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]
end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]
    print(status)
print(f"Final status: {status}")

### AgentCore Runtime 호출

이제 payload로 AgentCore Runtime을 호출할 수 있습니다.

에이전트가 `Get_calendar_events_today` 도구를 호출하고 3LO 흐름을 시작하는 것을 확인할 수 있습니다. 표시된 authorization URL을 클릭하거나 새 브라우저 session 또는 탭에 복사하여 user consent 흐름을 완료하세요.
권한 부여가 완료되면 credential provider `google-cal-provider`가 Google에서 `access_token`을 가져오고 도구 실행을 완료하여 Calendar 이벤트를 조회합니다.

<div style="text-align:left">
    <img src="images/invoke.png" width=75%"/>
</div>

In [ ]:
from oauth2_callback_server import (
    store_token_in_oauth2_callback_server,
    wait_for_oauth2_server_to_be_ready,
)

bearer_token = reauthenticate_user(cognito_config.get("client_id"))

oauth2_callback_server_cmd = [
    sys.executable,
    "oauth2_callback_server.py",
    "--region",
    region,
]
oauth2_callback_server_process = subprocess.Popen(oauth2_callback_server_cmd)

try:
    # OAuth2 callback server 시작
    successfully_started_oauth2_server = wait_for_oauth2_server_to_be_ready()
    if not successfully_started_oauth2_server:
        print(
            "Failed to start OAuth2 callback server to handle session binding "
            "(https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/oauth2-authorization-url-session-binding.html)"
        )
    else:
        store_token_in_oauth2_callback_server(bearer_token)
        invoke_response = agentcore_runtime.invoke(
            {"prompt": "What is in my agenda for today? Highlight the main events!"},
            bearer_token=bearer_token,
        )
        print(invoke_response)
finally:
    oauth2_callback_server_process.terminate()

## 선택 사항 - Streamlit App으로 에이전트 테스트

편리한 채팅 인터페이스를 제공하는 Streamlit 웹 애플리케이션으로 배포된 에이전트를 테스트할 수 있습니다. 이 디렉터리의 `chatbot_app_cognito.py` 파일은 다음 기능을 갖춘 웹 기반 챗봇을 생성합니다.

- `.bedrock_agentcore.yaml`에서 구성 자동 읽기
- Cognito 인증 제공
- 스트리밍 응답을 지원하는 현대적인 채팅 인터페이스 표시
- Google Calendar 액세스를 위한 3LO OAuth 흐름 처리

### Streamlit App 실행

Streamlit App은 여러 방식으로 실행할 수 있습니다.

#### 옵션 1: Jupyter Notebook에서 실행(현재 디렉터리)
- 아래 셀을 실행하여 이 Notebook에서 Streamlit App을 바로 시작합니다.
- 로그인: Cognito 설정에서 생성한 기본 테스트 사용자 자격 증명 testuser / MyPassword123!를 사용합니다.
- "Tell me a joke" 같은 간단한 prompt를 테스트합니다.
- `Get_calendar_events_today` 도구를 시작하는 "What is on my agenda for today?" 같은 prompt를 테스트합니다.
- 반환된 authorization URL을 클릭하거나 새 브라우저 탭 또는 창에 복사하여 user consent 흐름을 완료합니다.

In [ ]:
from chatbot_app_cognito import get_streamlit_url

# chatbot_app_cognito.py 파일이 있는 현재 디렉터리로 이동
notebook_dir = os.getcwd()

# Streamlit App 시작
print("Starting Streamlit app...")

oauth2_callback_server_cmd = [
    sys.executable,
    "oauth2_callback_server.py",
    "--region",
    region,
]
oauth2_callback_server_process = subprocess.Popen(oauth2_callback_server_cmd)

try:
    wait_for_oauth2_server_to_be_ready()

    # 현재 디렉터리에서 Streamlit 실행
    process = subprocess.Popen(
        [
            sys.executable,
            "-m",
            "streamlit",
            "run",
            "chatbot_app_cognito.py",
            "--server.port=8501",
            "--server.showEmailPrompt=false",
        ],
        cwd=notebook_dir,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    # 출력이 생성되는 즉시 표시
    for line in iter(process.stdout.readline, ""):
        if line:
            if "8501" in line:
                print("\n🎉 Streamlit app is ready!")
                streamlit_url = get_streamlit_url()
                print(f"\n🚀 Streamlit Application URL:\n{streamlit_url}\n")
                print("⚠️ To stop the app, interrupt the kernel or press Ctrl+C in the terminal")
                break

except KeyboardInterrupt:
    print("\nStreamlit app stopped.")
    oauth2_callback_server_process.terminate()
    process.terminate()
except Exception as e:
    print(f"Error starting Streamlit app: {e}")

#### 옵션 2: 터미널에서 실행

또는 터미널에서 Streamlit App을 실행할 수 있습니다.

```bash
# 현재 디렉터리로 이동
cd 06-workshops/03-AgentCore-identity/05-Outbound_Auth_3lo/

# Streamlit App 실행
python oauth2_callback_server.py -r <region> & streamlit run chatbot_app_cognito.py
```

#### Streamlit App 사용

1. **로그인**: Cognito 설정에서 생성한 기본 테스트 사용자 자격 증명 `testuser` / `MyPassword123!`를 사용합니다.
2. **채팅**: "What is in my agenda for today?" 또는 "Highlight main events from my agenda from today" 같은 질문을 합니다.
3. **OAuth 흐름**: 에이전트에 Google Calendar 액세스가 필요하면 authorization URL이 표시됩니다. URL을 클릭해 OAuth 흐름을 완료합니다.
4. **기능**: 앱에는 다음 기능이 포함됩니다.
   - 실시간 스트리밍 응답
   - 클릭 가능한 URL
   - 현대적인 채팅 인터페이스
   - 컨텍스트 인식
   - 유용한 메시지를 제공하는 오류 처리

앱은 `.bedrock_agentcore.yaml` 파일에서 모든 구성을 자동으로 읽으므로 방금 배포한 것과 동일한 AgentCore Runtime을 사용합니다.

## 정리(선택 사항)

- 이제 생성한 AgentCore Runtime을 정리하겠습니다.
- 아래 셀의 주석을 해제하고 실행하세요.

In [ ]:
# launch_result.ecr_uri, launch_result.agent_id, launch_result.ecr_uri.split('/')[1]

In [ ]:
# agentcore_control_client = boto3.client(
#     'bedrock-agentcore-control',
#     region_name=region
# )
# ecr_client = boto3.client(
#     'ecr',
#     region_name=region

# )

# runtime_delete_response = agentcore_control_client.delete_agent_runtime(
#     agentRuntimeId=launch_result.agent_id,

# )

# response = ecr_client.delete_repository(
#     repositoryName=launch_result.ecr_uri.split('/')[1],
#     force=True
# )

## 축하합니다!